# Stage C 03r — adaptive E100 trailblazer handoff

Run this companion only after notebook 03m reports that the adaptive E25 scale
gate passed. It leaves the frozen 03l notebook and C19 run unchanged.

The notebook creates an immutable E25-to-CURC input bundle, uploads it to the
`ecoeus` bucket, generates the CURC allocation brief, and prints the laptop
commands used to transfer the bundle to Alpine. Its import cell performs the
reverse path for E50/E75/E100 milestone bundles: CURC → laptop → GCS → Colab →
Drive. No permanent Google credential is stored on CURC.


In [ ]:
# @title 1. Project and Drive configuration
PROJECT_ID='divine-tempo-502518-j4' # @param {type:"string"}
BUCKET='ecoeus' # @param {type:"string"}
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC' # @param {type:"string"}
GIT_REF='main' # @param {type:"string"}
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
STAGE_C_RUNNER='seqtrainer-titans-stage-c-colab-run'  # handoff operations themselves do not train

from pathlib import Path
from google.colab import auth, drive
import json, os, shutil, subprocess, sys

mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
auth.authenticate_user()
subprocess.run(['gcloud','config','set','project',PROJECT_ID],check=True)
repo=Path('/content/SeqTrainer-trailblazer-support')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',f'origin/{GIT_REF}'],check=True)
launcher=repo/'scripts/stage_c_c20_curc.py'
control=repo/'scripts/stage_c_e25_control_gcp.py'
adaptive_gcp=repo/'scripts/stage_c_adaptive_e100_gcp.py'
for path in (launcher,control,adaptive_gcp):
    if not path.is_file(): raise FileNotFoundError(path)
print('Support checkout:',subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())


In [ ]:
# @title 2. Export the passed E25 parent and upload the immutable CURC input
exports=Path('/content/stage_c_trailblazer_inputs')
subprocess.run([sys.executable,str(launcher),'export','--drive-root',DRIVE_ROOT,
 '--repo-root',str(repo),'--output',str(exports)],check=True)
bundles=sorted(path.parent for path in exports.glob('*/TRAILBLAZER_INPUT_MANIFEST.json'))
if not bundles: raise RuntimeError('No input bundle was produced')
INPUT_BUNDLE=bundles[-1]
INPUT_MANIFEST=json.loads((INPUT_BUNDLE/'TRAILBLAZER_INPUT_MANIFEST.json').read_text())
INPUT_URI=f"gs://{BUCKET}/stage-c/trailblazer/inputs/{INPUT_MANIFEST['bundle_id']}"
published=f"{INPUT_URI}/PUBLISHED_TRAILBLAZER_INPUT_MANIFEST.json"
probe=subprocess.run(['gcloud','storage','ls',published],text=True,capture_output=True)
if probe.returncode==0:
    raise RuntimeError(f'Immutable GCS input already exists: {published}')
subprocess.run(['gcloud','storage','rsync','--recursive',str(INPUT_BUNDLE),INPUT_URI],check=True)
subprocess.run(['gcloud','storage','cp','--if-generation-match=0',
 str(INPUT_BUNDLE/'TRAILBLAZER_INPUT_MANIFEST.json'),published],check=True)
brief=Path('/content/CURC_ASCENT_ALLOCATION_BRIEF.md')
subprocess.run([sys.executable,str(launcher),'allocation-brief','--output',str(brief)],check=True)
print(brief.read_text())
print('\nINPUT URI:',INPUT_URI)
print('\nON YOUR LAPTOP (not CURC):')
print(f"gcloud storage cp --recursive {INPUT_URI} ./stage-c-curc-input")
print("rsync -av --partial ./stage-c-curc-input/ <identikey>@login.rc.colorado.edu:/projects/<identikey>/stage-c/input/")
print('\nThe CURC preflight/pilot/submit commands are documented in docs/titans_stage_c/TRAILBLAZER_E100_RUNBOOK.md.')


In [ ]:
# @title 3. GCP no-memory E25 commands (only after the same gate passed)
print('Pilot preflight:')
print(' '.join([sys.executable,str(control),'preflight','--project',PROJECT_ID,'--bucket',BUCKET,
 '--bundle',str(INPUT_BUNDLE),'--region','auto','--phase','pilot']))
print('\nAfter downloading GCP_PILOT_REPORT.json, production preflight:')
print(' '.join([sys.executable,str(control),'preflight','--project',PROJECT_ID,'--bucket',BUCKET,
 '--bundle',str(INPUT_BUNDLE),'--region','auto','--phase','production',
 '--pilot-report','/content/GCP_PILOT_REPORT.json']))
print('Provision uses the same arguments with `provision` in place of `preflight`.')

# @title 4. Prepare adaptive GCP replica or CURC failover
ADAPTIVE_ROLE='gcp-replica' # @param ["gcp-replica", "gcp-failover"]
print('Production requires a private GOOGLE_CLOUD_CREDIT_ACTIVATION.json created only after the award is visible.')
command=[sys.executable,str(adaptive_gcp),'preflight','--project',PROJECT_ID,'--bucket',BUCKET,
 '--bundle',str(INPUT_BUNDLE),'--role',ADAPTIVE_ROLE,'--region','auto','--phase','production',
 '--credit-activation','/content/GOOGLE_CLOUD_CREDIT_ACTIVATION.json']
if ADAPTIVE_ROLE=='gcp-failover':
    print('First export CURC recovery:')
    print('python scripts/stage_c_c20_curc.py export-recovery --run-dir <curc-run> --output <recovery-output>')
    print('Transfer CURC -> laptop -> GCS -> Colab and add --recovery /content/curc-recovery below.')
    print('Production also requires --failover-authorization /content/FAILOVER_AUTHORIZATION.json.')
print('\nAdaptive production preflight:')
print(' '.join(command))
print('Use `provision` in place of `preflight` only after this command passes.')


In [ ]:
# @title 5. Import one returned CURC or GCP milestone into Drive
MILESTONE='e50' # @param ["e50", "e75", "e100"]
MILESTONE_URI='' # @param {type:"string"}
if not MILESTONE_URI.startswith('gs://'):
    raise ValueError('Upload the CURC milestone bundle from your laptop to GCS, then paste its gs:// URI.')
download=Path('/content/curc-return')/MILESTONE
if download.exists(): shutil.rmtree(download)
download.mkdir(parents=True)
subprocess.run(['gcloud','storage','rsync','--recursive',MILESTONE_URI,str(download)],check=True)
verified=json.loads(subprocess.check_output([sys.executable,str(launcher),'verify-milestone','--bundle',str(download)],text=True))
if verified['milestone'] != MILESTONE: raise ValueError('Returned bundle has the wrong milestone label')
trajectory=verified.get('trajectory_id','adaptive-seed-20260751')
role=verified.get('execution_role','curc-primary')
if trajectory=='adaptive-seed-20260751' and role=='curc-primary':
    target=Path(DRIVE_ROOT)/'runs/c20_v3_medium_adaptive_e100_increment/milestones'/MILESTONE
else:
    target=Path(DRIVE_ROOT)/'runs/stage_c_adaptive_trajectories'/trajectory/role/'milestones'/MILESTONE
if target.exists():
    existing=json.loads((target/'MILESTONE_MANIFEST.json').read_text())
    if existing['checkpoint']['sha256'] != verified['checkpoint']['sha256']:
        raise RuntimeError(f'Refusing to replace a different Drive milestone: {target}')
    print('Identical milestone is already published:',target)
else:
    partial=target.with_name(target.name+'.partial')
    if partial.exists(): raise RuntimeError(f'Remove or inspect stale partial publication: {partial}')
    partial.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(download,partial)
    import hashlib
    def sha(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
        return h.hexdigest()
    if sha(partial/'latest.pt') != verified['checkpoint']['sha256']:
        raise RuntimeError('Drive copy checksum mismatch')
    if sha(partial/'model.pt') != verified['checkpoint']['sha256']:
        raise RuntimeError('Drive model.pt checksum mismatch')
    os.replace(partial,target)
pointer={
 'format_version':1,'milestone':MILESTONE,'checkpoint_path':str(target/'latest.pt'),
 'model_path':str(target/'model.pt'),'trajectory_id':trajectory,'execution_role':role,
 'scientific_seed':verified.get('scientific_seed'),
 'sha256':verified['checkpoint']['sha256'],
 'optimizer_step':verified['checkpoint']['optimizer_step'],
 'processed_bases':verified['checkpoint']['processed_bases'],
}
(target/'MODEL_POINTER.json').write_text(json.dumps(pointer,indent=2,sort_keys=True)+'\n')
print('\nPASTE THIS CHECKPOINT PATH INTO AN EVALUATION NOTEBOOK:')
print(target/'model.pt')
print(json.dumps(pointer,indent=2))
